# Step 3 — Dual-Process MuJoCo Control (VLA + ESN)

**Strict dual-process integration only** — no dataset oracle or benchmark video here.

- **Process A (VLA):** UnifoLM-VLA-Base @ ~2 Hz → 25-step action chunk → EE→joint IK
- **Process B (MuJoCo + ESN):** 100 Hz physics + CUDA ESN readout

Lock-free `multiprocessing.Array` registers bypass the Python GIL.

Start with `MOCK=True`. For **dataset replay, cloth grasp, and video** use **Step 4**:
`notebooks/step4_mujoco_evaluation.ipynb`

```bash
MUJOCO_GL=egl python3 -m src.step3_dual_thread_mujoco --mock --duration_s 5
```

In [ ]:
from pathlib import Path
import os
import sys

# Headless MuJoCo rendering on servers without DISPLAY.
os.environ.setdefault("MUJOCO_GL", "egl")

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    RESEARCH_DIR = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "src" / "step3_dual_thread_mujoco.py").is_file():
    RESEARCH_DIR = NOTEBOOK_DIR
else:
    RESEARCH_DIR = NOTEBOOK_DIR / "research_summer_2026" / "research"
    if not RESEARCH_DIR.is_dir():
        RESEARCH_DIR = NOTEBOOK_DIR / "research"

RESEARCH_DIR = RESEARCH_DIR.resolve()
assert (RESEARCH_DIR / "src").is_dir(), f"src package not found under: {RESEARCH_DIR}"

os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step3_dual_thread'}")

In [ ]:
# ── Configuration (dual-process only) ───────────────────────────
MOCK = True                    # True = mock VLA (no 18 GB download)
DURATION_S = 10.0
CONTROL_HZ = 100.0
VLA_HZ = 2.0
PROFILE = False
PROFILE_STEPS = 200
DEVICE = "cuda"
RECORD_VIDEO = False           # optional debug; full video → Step 4
VIDEO_FPS = 60.0

INSTRUCTION = "Clean the table with the cloth."
UNNORM_KEY = "g1_wipe_table"
INIT_EPISODE = 0
MJCF_PATH = None
ESN_CHECKPOINT = None

print(f"MOCK={MOCK} | duration={DURATION_S}s | control={CONTROL_HZ:.0f} Hz | VLA={VLA_HZ:.0f} Hz")
print(f"record_video={RECORD_VIDEO} (use step4_mujoco_evaluation for benchmark video)")

In [ ]:
import json
import logging
import multiprocessing as mp

import torch

from src.paths import results_path
from src.step3_dual_thread_mujoco import (
    DualProcessConfig,
    DualProcessController,
    MAX_DURATION_S,
    MAX_STEP_MS_THRESHOLD,
    load_esn_checkpoint_metadata,
    print_run_summary,
    resolve_esn_checkpoint,
    resolve_mjcf_path,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required (target: Tesla V100).")
if DURATION_S > MAX_DURATION_S:
    raise ValueError(f"DURATION_S={DURATION_S} exceeds MAX_DURATION_S={MAX_DURATION_S}")

print(f"Device: cuda ({torch.cuda.get_device_name(torch.device(DEVICE))})")

mjcf = resolve_mjcf_path(MJCF_PATH)
ckpt = resolve_esn_checkpoint(ESN_CHECKPOINT)
esn_meta = load_esn_checkpoint_metadata(ckpt)
out_dir = results_path("step3_dual_thread")
video_path = out_dir / "table_wipe_benchmark.mp4"

print(f"MJCF: {mjcf}")
print(f"ESN checkpoint (Step 2): {ckpt}")
if esn_meta.get("metrics"):
    m = esn_meta["metrics"]
    print(
        f"  Step 2 metrics: MSE={m['mse']:.2e} jerk={m['jerk']:.2e} "
        f"α={m['leaky_rate']:.2f} λ={m['ridge_alpha']:.1e} "
        f"dataset={esn_meta.get('dataset_id', '?')}"
    )
print(f"Video output: {video_path} ({DURATION_S:.0f}s max @ {VIDEO_FPS:.0f} fps)")

In [ ]:
mp.set_start_method("spawn", force=True)

config = DualProcessConfig(
    mjcf_path=mjcf,
    esn_checkpoint=str(ckpt),
    mock=MOCK,
    duration_s=DURATION_S,
    control_hz=CONTROL_HZ,
    vla_hz=VLA_HZ,
    instruction=INSTRUCTION,
    device=DEVICE,
    profile=PROFILE,
    profile_steps=PROFILE_STEPS,
    record_video=RECORD_VIDEO,
    video_path=video_path if RECORD_VIDEO else None,
    video_fps=VIDEO_FPS,
    unnorm_key=UNNORM_KEY,
    init_episode=INIT_EPISODE,
    use_wipe_table_scene=True,
)

print(f"Starting dual-process run (mock={MOCK}) for {DURATION_S:.1f}s ...")
controller = DualProcessController(config)
stats = controller.run()

report = {
    "architecture": "multiprocessing",
    "task": "g1_wipe_table",
    "mock_vla": MOCK,
    "mjcf": str(mjcf),
    "esn_checkpoint": str(ckpt),
    "duration_s": DURATION_S,
    "control_hz_target": CONTROL_HZ,
    "vla_hz_target": VLA_HZ,
    "steps": stats.steps,
    "mean_step_ms": stats.mean_step_ms,
    "max_step_ms": stats.max_step_ms,
    "max_step_ms_steady": stats.max_step_ms_steady,
    "p99_step_ms": stats.p99_step_ms,
    "achieved_esn_hz": stats.esn_hz,
    "vla_ticks": stats.vla_ticks,
    "vla_register_sequence": stats.vla_seq_final,
    "gil_bypass_ok": stats.gil_bypass_ok,
    "max_step_ms_threshold": MAX_STEP_MS_THRESHOLD,
    "video_path": stats.video_path,
}

report_path = out_dir / "dual_thread_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print_run_summary(
    stats,
    control_hz=CONTROL_HZ,
    vla_hz=VLA_HZ,
    report_path=report_path,
    profile=PROFILE,
)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Video, display

summary = pd.DataFrame([report]).T
summary.columns = ["value"]
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(
    ["Target ESN Hz", "Achieved ESN Hz", "Target VLA Hz", "VLA ticks"],
    [CONTROL_HZ, stats.esn_hz, VLA_HZ, stats.vla_ticks],
    color=["#9E9E9E", "#4CAF50", "#9E9E9E", "#2196F3"],
)
axes[0].axhline(CONTROL_HZ, color="#F44336", ls="--", lw=1.5, label=f"{CONTROL_HZ:.0f} Hz target")
axes[0].set_ylabel("Hz / count")
axes[0].set_title("Phase 3 throughput summary")
axes[0].legend()
for b in bars:
    axes[0].text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        f"{b.get_height():.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

lat_colors = [
    "#4CAF50" if stats.max_step_ms_steady < MAX_STEP_MS_THRESHOLD else "#F44336",
    "#2196F3",
    "#9C27B0",
]
axes[1].bar(
    ["Steady max (ms)", "Mean step (ms)", "P99 (ms)"],
    [stats.max_step_ms_steady, stats.mean_step_ms, stats.p99_step_ms],
    color=lat_colors,
)
axes[1].axhline(MAX_STEP_MS_THRESHOLD, color="#F44336", ls="--", lw=1.5, label=f"{MAX_STEP_MS_THRESHOLD:.0f} ms GIL threshold")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Control-loop latency (GIL-free)")
axes[1].legend()

plt.tight_layout()
fig_path = out_dir / "dual_thread_summary.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Figure saved: {fig_path}")

if stats.video_path and Path(stats.video_path).is_file():
    from IPython.display import HTML, Video, display

    print(f"Benchmark video: {stats.video_path}")
    # H.264 MP4 for browser playback (avc1); fall back to HTML5 <video> tag.
    try:
        display(Video(stats.video_path, embed=True, width=640, html_attributes="controls loop"))
    except Exception:
        display(
            HTML(
                f'<video width="640" controls loop>'
                f'<source src="{stats.video_path}" type="video/mp4">'
                f"</video>"
            )
        )